# Projet Hadoop — Notebook explicatif du modèle de clustering

Ce notebook est pensé pour **quelqu’un qui ne connaît pas bien le machine learning**.

Il explique :
- ce que fait le projet ;
- quelles données sont utilisées ;
- comment elles sont préparées ;
- comment le modèle est entraîné ;
- comment on lit les résultats.

## Idée générale

On a un dataset de **séries TV**.

On veut regrouper les séries qui se ressemblent selon plusieurs caractéristiques :
- popularité ;
- nombre de votes ;
- décennie ;
- genres ;
- longueur de la série ;
- densité de contenu.

Le modèle utilisé est un **K-Means**.

## Important

Ici, on ne prédit pas une classe connue à l’avance.

On fait du **clustering** :

> le modèle regroupe automatiquement les séries dans des groupes similaires.

## Schéma global de la pipeline

```text
CSV bruts
   ↓
Cleaning Spark / HDFS
   ↓
shows_clean
   ↓
Feature engineering
   ↓
VectorAssembler
   ↓
StandardScaler
   ↓
K-Means
   ↓
shows_clustered
   ↓
Analyse / Recommandation / Visualisation
```

### Lecture simple

- **Cleaning** : on nettoie les données.
- **Feature engineering** : on crée des colonnes plus utiles.
- **VectorAssembler** : on rassemble les colonnes en un seul vecteur.
- **StandardScaler** : on met les variables à la même échelle.
- **K-Means** : on forme des groupes.

## 1. Initialisation de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as spark_round
from pyspark.ml.clustering import KMeansModel
from pyspark.ml.feature import StandardScalerModel
import pandas as pd
import json
import matplotlib.pyplot as plt

HDFS_PATH = "hdfs://localhost:9000/user/user/data"

spark = (
    SparkSession.builder
    .appName("Notebook explicatif - Projet séries TV")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark est initialisé.")
print("Chemin HDFS utilisé :", HDFS_PATH)

## 2. Chargement du résultat final du clustering

Le fichier `shows_clustered` contient les séries après le passage dans le modèle.

Chaque ligne correspond à une série avec son `cluster_id`.

In [ ]:
df_clustered = spark.read.parquet(f"{HDFS_PATH}/shows_clustered")

print("Nombre total de séries utilisées :", df_clustered.count())
df_clustered.printSchema()
df_clustered.show(10, truncate=False)

## 3. Rapport du modèle final

Le script `clustering.py` produit un fichier `outputs/model_report.json`.

Ce rapport résume :
- le nombre de clusters `k` ;
- la seed ;
- le score de silhouette ;
- le nombre de séries ;
- le nombre de features ;
- la taille de chaque cluster.

### Rappel : silhouette
- proche de **1** : bonne séparation ;
- proche de **0** : groupes qui se chevauchent ;
- négatif : mauvais clustering.

In [ ]:
with open("outputs/model_report.json", encoding="utf-8") as f:
    report = json.load(f)

report

### Ce qu’il faut retenir

Dans notre cas :
- modèle final : **K-Means** ;
- `k = 8` ;
- `seed = 42` ;
- silhouette ≈ **0,2707**.

Ce n’est pas parfait, mais c’est exploitable pour le projet.

## 4. Les variables utilisées par le modèle

In [ ]:
with open("outputs/feature_cols.json", encoding="utf-8") as f:
    feature_cols = json.load(f)

print("Nombre de features :", len(feature_cols))
feature_cols

### Explication intuitive

Les features sont les colonnes que le modèle utilise pour comparer les séries.

Exemples :
- `popularity` : popularité ;
- `vote_count` : nombre de votes ;
- `decade` : décennie ;
- `Comedy`, `Drama`, `Crime`... : genres ;
- `serie_length` : longueur de la série ;
- `vote_quality` : qualité pondérée ;
- `content_density` : épisodes par saison.

### Pourquoi normaliser ?

Parce que les colonnes n’ont pas la même échelle.
Sans normalisation, certaines variables prendraient trop d’importance.

## 5. Taille des clusters

In [ ]:
cluster_sizes = (
    df_clustered.groupBy("cluster_id")
    .count()
    .orderBy("cluster_id")
)
cluster_sizes.show()

In [ ]:
cluster_sizes_pd = cluster_sizes.toPandas()

plt.figure(figsize=(9, 4))
plt.bar(cluster_sizes_pd["cluster_id"].astype(str), cluster_sizes_pd["count"])
plt.xlabel("Cluster")
plt.ylabel("Nombre de séries")
plt.title("Répartition des séries par cluster")
plt.show()

### Lecture du graphe

Chaque barre représente un cluster.

Plus une barre est haute, plus le cluster contient de séries.

On voit ici que les clusters sont **déséquilibrés** :
- certains sont très gros ;
- d’autres sont plus spécialisés.

## 6. Résumé des clusters

In [ ]:
summary = (
    df_clustered.groupBy("cluster_id")
    .agg(
        count("*").alias("nb_series"),
        spark_round(avg("popularity"), 2).alias("popularite_moyenne"),
        spark_round(avg("vote_quality"), 2).alias("qualite_vote_moyenne"),
        spark_round(avg("vote_count"), 2).alias("votes_moyens"),
        spark_round(avg("serie_length"), 2).alias("longueur_moyenne"),
        spark_round(avg("content_density"), 2).alias("episodes_par_saison")
    )
    .orderBy("cluster_id")
)

summary.show(truncate=False)

### Comment lire ce tableau ?

Pour chaque cluster :
- `nb_series` : nombre de séries ;
- `popularite_moyenne` : popularité moyenne ;
- `qualite_vote_moyenne` : note pondérée moyenne ;
- `votes_moyens` : nombre moyen de votes ;
- `longueur_moyenne` : longueur moyenne ;
- `episodes_par_saison` : densité moyenne de contenu.

Cela aide à décrire les groupes simplement.

## 7. Comparaison des expériences selon K

In [ ]:
results = pd.read_csv("experiments/results.csv")
results

### Pourquoi tester plusieurs K ?

Dans K-Means, il faut choisir le nombre de groupes à créer.

Comme on ne connaît pas ce nombre à l’avance, on teste plusieurs valeurs :
- 3
- 5
- 8
- 10
- 12
- 15

In [ ]:
mean_results = results.groupby("k", as_index=False).agg(
    silhouette=("silhouette", "mean"),
    cout=("cout", "mean")
)

mean_results

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(mean_results["k"], mean_results["silhouette"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Silhouette moyenne")
plt.title("Silhouette moyenne selon K")
plt.show()

### Lecture du graphe de silhouette

Ce graphe montre la qualité moyenne du clustering selon `K`.

On ne choisit pas forcément le plus grand score brut :
il faut aussi regarder la stabilité et la taille des clusters.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(mean_results["k"], mean_results["cout"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Coût moyen")
plt.title("Coût K-Means selon K")
plt.show()

### Lecture du graphe de coût

Le coût diminue souvent quand `K` augmente.

Mais un coût plus faible ne suffit pas :
si des clusters deviennent minuscules ou peu lisibles, ce n’est pas forcément un bon choix.

### Pourquoi `K = 8` ?

On a retenu `K = 8` avec `seed = 42` car :
- la silhouette est correcte ;
- les clusters restent exploitables ;
- on évite certains résultats aberrants.

## 8. Chargement des modèles sauvegardés

In [ ]:
scaler_model = StandardScalerModel.load(f"{HDFS_PATH}/models/scaler_model")
kmeans_model = KMeansModel.load(f"{HDFS_PATH}/models/kmeans_model")

print("Scaler chargé avec succès.")
print("K-Means chargé avec succès.")
print("Nombre de clusters :", kmeans_model.getK())

### Pourquoi cette étape est importante ?

Elle montre que :
- le modèle a bien été entraîné ;
- il a bien été sauvegardé ;
- on peut le recharger plus tard sans réentraîner.

## 9. Cas de test

In [ ]:
series_test = ["Lie to Me", "Sherlock", "Breaking Bad", "Friends", "The Mentalist"]

df_clustered.filter(col("name").isin(series_test)) \
    .select("name", "cluster_id", "popularity", "vote_quality", "vote_count") \
    .orderBy("name") \
    .show(truncate=False)

### Comment lire les cas de test ?

On observe dans quel cluster tombent quelques séries connues.

Attention :
le modèle ne comprend pas le scénario ou le sens profond d’une série.

Il compare surtout des **métadonnées structurées**.

## 10. Bilan final

### Ce qui a été fait

- nettoyage des données ;
- création des features ;
- normalisation avec `StandardScaler` ;
- entraînement de plusieurs K-Means ;
- comparaison des résultats ;
- choix d’un modèle final ;
- sauvegarde du scaler et du K-Means ;
- rechargement du modèle ;
- production des clusters finaux.

### Résultat retenu

- **Modèle :** K-Means
- **Nombre de clusters :** 8
- **Seed :** 42
- **Nombre de séries :** 53 532
- **Nombre de features :** 26
- **Silhouette :** environ 0,2707

### Conclusion

Le projet montre bien une vraie pipeline de machine learning avec Hadoop / Spark.

Même si les résultats ne sont pas parfaits, le travail principal est bien là :
- entraînement ;
- comparaison ;
- choix du modèle ;
- sauvegarde ;
- rechargement ;
- interprétation.

In [ ]:
# À exécuter à la fin du notebook
spark.stop()